# 03 - Feature engineering

Row-level business features are written in SQL (`customer_features` view) so a reviewer can read
the definition next to the data. Anything that must be *fitted* (scaling, one-hot encoding)
lives in a scikit-learn `Pipeline`, fitted on training folds only.

| Brief asked for | What the Telco data allows | Feature |
|---|---|---|
| Customer_Lifetime_Months | tenure | `tenure` (+ `tenure_bucket` for BI) |
| Average_Monthly_Spend | total / tenure | `avg_monthly_spend` |
| Usage_Change_Percentage | no usage history - bill trend is the closest honest proxy | `spend_trend_ratio` = current bill / lifetime average |
| Service_Count | 9 services | `service_count`, `charge_per_service` |
| Support_Calls_Per_Month, Payment_Delay_Rate | **not in the dataset** - not fabricated | - |
| (extra) | stickiness of the security + support pair | `has_protection_bundle` |
| (extra) | household | `family_account` |

In [1]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from IPython.display import Image, display
from src import config
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
feats = pd.read_csv(config.FEATURES_CSV)
eng = ["avg_monthly_spend", "spend_trend_ratio", "service_count", "charge_per_service",
       "has_protection_bundle", "family_account"]
feats[["customer_id", "tenure", "monthly_charges", "total_charges"] + eng].head()

,customer_id,tenure,monthly_charges,total_charges,avg_monthly_spend,spend_trend_ratio,service_count,charge_per_service,has_protection_bundle,family_account
0,0002-ORFBO,9,65.6,593.30,65.92,0.995,5,13.12,0,1
1,0003-MKNFE,9,59.9,542.40,60.27,0.994,4,14.97,0,0
2,0004-TLHLJ,4,73.9,280.85,70.21,1.053,3,24.63,0,0
3,0011-IGKFF,13,98.0,1237.85,95.22,1.029,6,16.33,0,1
4,0013-EXCHZ,3,83.9,267.40,89.13,0.941,4,20.98,0,1


In [2]:
sql = (config.SQL_DIR / "01_create_schema.sql").read_text()
print(sql[sql.index("CREATE VIEW customer_features"):])

CREATE VIEW customer_features AS
SELECT
    c.*,

    -- Average bill across the whole life of the account. A customer who has
    -- not been billed yet (tenure 0) is, by definition, on their current bill.
    COALESCE(ROUND(c.total_charges / NULLIF(c.tenure, 0), 2),
             c.monthly_charges)                                AS avg_monthly_spend,

    -- > 1 means the customer is currently paying more than they historically
    -- did: a price rise, an up-sell, or a promo that has just expired.
    -- Stands in for "usage change %": the dataset has no usage history.
    COALESCE(ROUND(c.monthly_charges /
             NULLIF(c.total_charges / NULLIF(c.tenure, 0), 0), 3),
             1.0)                                              AS spend_trend_ratio,

    (CASE WHEN c.phone_service     = 'Yes' THEN 1 ELSE 0 END +
     CASE WHEN c.multiple_lines    = 'Yes' THEN 1 ELSE 0 END +
     CASE WHEN c.internet_service <> 'No'  THEN 1 ELSE 0 END +
     CASE WHEN c.online_security   = 'Yes

## Do the engineered features separate churners?

In [3]:
def rate_by(col, bins=None, labels=None):
    s = pd.cut(feats[col], bins=bins, labels=labels) if bins is not None else feats[col]
    return feats.groupby(s, observed=True)["churn"].agg(customers="size", churn_rate="mean").round(3)

display(rate_by("has_protection_bundle"))
display(rate_by("family_account"))
display(rate_by("spend_trend_ratio", [0, 0.95, 1.05, 10], ["bill fell", "flat", "bill rose"]))
display(rate_by("charge_per_service", [0, 20, 30, 45], ["<=$20", "$20-30", "$30-45"]))

,customers,churn_rate
has_protection_bundle,,
0,5944,0.298
1,1099,0.090


,customers,churn_rate
family_account,,
0,3280,0.342
1,3763,0.198


,customers,churn_rate
spend_trend_ratio,,
bill fell,637,0.261
flat,5670,0.263
bill rose,736,0.284


,customers,churn_rate
charge_per_service,,
<=$20,5183,0.213
$20-30,1627,0.385
$30-45,233,0.592


- **Security + support bundle**: churn drops from 29.8% to 9.0%.
- **Family account** (partner or dependents): 19.8% vs. 34.2% for single-person accounts.
- **Price per service**: 21% churn at <=$20 per service, 39% at $20-30, 59% above $30 - the
  "not getting value for money" profile.
- **Bill trend** barely separates churners on its own (26-28% in every band). It is the weakest
  engineered feature; the ablation below is the honest test of whether the set as a whole helps.

## Deliberately left out of the model

`is_month_to_month`, `is_electronic_check`, `is_auto_payment`, `is_new_customer` and
`tenure_bucket` are kept in the SQL view for dashboards but **not** fed to the model: they are
recodings of columns that are already one-hot encoded, and duplicates would split SHAP credit
between two columns that mean the same thing.

## Did it help? Ablation with the champion model

In [4]:
report = json.loads(config.METRICS_JSON.read_text())
pd.DataFrame(report["feature_ablation"]).round(4)

,feature_set,n_features,cv_roc_auc,cv_roc_auc_std,cv_pr_auc
0,Raw columns only,19,0.8487,0.0162,0.6625
1,Raw + engineered features,25,0.8485,0.0164,0.6608


**Honest result: no gain.** CV ROC-AUC is 0.8487 with the raw columns and 0.8485 with the
engineered features added - identical within noise (std ~0.016). The champion is a boosted-tree
model with depth-2 trees, and it already learns combinations such as spend per month or price
per service from the raw columns.

The features stay in the SQL layer because they make the BI segments and the explanations more
readable ("pays $32 per service - more than almost every other customer" is something an account
manager can act on), not because they add accuracy. Datasets with real behavioural history
(support calls, usage trends, payment delays) are where feature engineering usually pays off;
the Telco extract does not contain any.